In [8]:
import streamlit as st
import pandas as pd
import os
import configparser
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F

config_path = os.path.join(os.environ['USERPROFILE'], '.snowsql', 'config')
config = configparser.ConfigParser()
config.read(config_path)

try:
    account = config['connections.example']['accountname']
    user = config['connections.example']['username']
    password = config['connections.example']['password']
except KeyError as e:
    print(f'Error: {e}')

connection_parameters = {
    "account": account,
    "user": user,
    "password": password,
    "warehouse": "COMPUTE_WH",
    "database": "DEVMOON_SAMPLE",
    "schema": "PUBLIC",
}

try:
    session = Session.builder.configs(connection_parameters).create()
    print('Conexión exitosa con Snowpark.')
except Exception as e:
    print(f'Error: {e}')

Conexión exitosa con Snowpark.


In [ ]:
orders_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS')
customer_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER')
nation_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION')
region_table = session.table('SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.REGION')


In [ ]:
usa_region_key = region_table.filter(
    F.col('R_NAME') == 'AMERICA').select('R_REGIONKEY')
usa_nation_key = nation_table.filter(
    (F.col('N_NAME') == 'UNITED STATES') & (F.col('N_REGIONKEY').isin(usa_region_key))).select('N_NATIONKEY')
usa_customer_key = customer_table.filter(
    F.col('C_NATIONKEY').isin(usa_nation_key)).select('C_CUSTKEY')
usa_orders = orders_table.filter(F.col('O_CUSTKEY').isin(usa_customer_key))

In [13]:
daily_salea_sndf = usa_orders.group_by(
    F.to_date(F.col('O_ORDERDATE')).alias('ORDER_DATE')
).agg(F.sum('O_TOTALPRICE').alias('TOTAL_SALES'))

daily_salea_sndf.count()

2406

In [16]:
daily_salea_pdf = daily_salea_sndf.to_pandas()
daily_salea_pdf = daily_salea_pdf.sort_values(by = 'ORDER_DATE')

In [17]:
import plotly.express as px

In [19]:
px.line(daily_salea_pdf, x = 'ORDER_DATE', y = 'TOTAL_SALES',
        title='Ventas Diarias totales en Estados Unidos.')